###Referenz ⇒ Canonical Form 
- pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
- bei SAbPred: SCALOP in Submission form die Datei hochladen 
- results: für jedes CDR (H1, H2, L1, L2, L3) erkennt er CDR Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
- (L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
- muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
- dann der vergleich mit V-measure

'''# SCALOP Installation und Setup Anleitung
# 1. Projekt klonen (wenn noch nicht gemacht)
git clone https://github.com/oxpig/SCALOP.git

# 2. Conda-Umgebung erstellen
conda create -n scalop-env python=3.8 -y

# 3. Conda initialisieren (falls noch nie gemacht)
conda init
# dann Terminal neu starten
exit
# dann Terminal wieder öffnen

# 4. Umgebung aktivieren
conda activate scalop-env

# 5. Abhängigkeiten installieren
conda install -c bioconda numpy biopython -y
conda install -c bioconda hmmer -y   # funktioniert nur in WSL-Umgebung, nicht in Windows direkt! Wird für assign benötigt

# 6. SCALOP lokal installieren
pip install ./SCALOP

# 7. Kernel registrieren (für Jupyter-Notebook))
pip install ipykernel
python -m ipykernel install --user --name scalop-env --display-name "Python (scalop-env)"
# dann Jupyter Notebook öffnen und den Kernel "Python (scalop-env)" auswählen. Jetzt kann man SCALOP in Jupyter Notebooks verwenden.'''

In [ ]:
import pandas as pd
from scalop.predict import assign

# Datei einlesen
df = pd.read_csv("../data/ab_ag_vseqs.tsv", sep="\t")

results_all = []

for idx, row in df.iterrows():
    pdb_id = row['pdb']
    antigen_name = row['antigen_name']
    vh_seq = row['VH']
    vl_seq = row['VL']

    # Ergebnisse Dictionary vorbereiten
    data_row = {
        "PDB_ID": pdb_id,
        "antigen_name": antigen_name
    }

    # Heavy Chain nur H1, H2
    try:
        results_vh = assign(vh_seq, scheme="chothia", definition="chothia")[0]['outputs']
        for cdr in ['H1', 'H2']:
            data_row[f"SEQ_{cdr}"] = results_vh[cdr][1] if cdr in results_vh else None
            data_row[f"CF_{cdr}"]  = results_vh[cdr][2] if cdr in results_vh else None
    except:
        data_row.update({f"SEQ_{cdr}": None for cdr in ['H1', 'H2']})
        data_row.update({f"CF_{cdr}": None for cdr in ['H1', 'H2']})

    # Light Chain nur L1, L2, L3
    try:
        results_vl = assign(vl_seq, scheme="chothia", definition="chothia")[0]['outputs']
        for cdr in ['L1', 'L2', 'L3']:
            data_row[f"SEQ_{cdr}"] = results_vl[cdr][1] if cdr in results_vl else None
            data_row[f"CF_{cdr}"]  = results_vl[cdr][2] if cdr in results_vl else None
    except:
        data_row.update({f"SEQ_{cdr}": None for cdr in ['L1', 'L2', 'L3']})
        data_row.update({f"CF_{cdr}": None for cdr in ['L1', 'L2', 'L3']})

    results_all.append(data_row)

# In DataFrame und speichern
df_out = pd.DataFrame(results_all)
df_out.to_csv("canonical_forms_per_antibody_full.csv", index=False)
print("Fertig! Datei gespeichert als canonical_forms_per_antibody_full.csv")

Fertig! Datei gespeichert als canonical_forms_per_antibody_full.csv


In [ ]:
import pandas as pd

# Datei einlesen
df = pd.read_csv("../data/canonical_forms_per_antibody_full.csv")

# Schritt 1: Alle Zeilen mit mindestens einem Missing Value entfernen
df_clean = df.dropna()

# Schritt 2: Doppelte Einträge basierend auf den SEQ_* Spalten entfernen
cdr_seq_columns = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
n_before = len(df_clean)

df_unique = df_clean.drop_duplicates(subset=cdr_seq_columns, keep='first')
n_after = len(df_unique)
duplicates_removed = n_before - n_after

# Speichern
df_unique.to_csv("ab_ag_canonical_forms.csv", index=False)

print(f"Fertig! Es wurden {duplicates_removed} Duplikate entfernt und alle N/A entfernt.")
print(f"Neue Datei hat {n_after} eindeutige PDBs.")

In [ ]:
import sys
print(sys.executable)

c:\Users\avdh3\OneDrive\Dokumente\GitHub\group04-team04\.conda\python.exe


In [ ]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path
        

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None